# Modelo EGARCH: Entrenamiento y Evaluación

In [2]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.egarch_wrapper import EGARCHWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "EGARCH"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- El protocolo Walk-Forward seleccionó al EGARCH(1,3) con distribución $t$ de Student como la mejor configuración.
- La prueba de Razón de Verosimilitud ratificó la elección de la distribución $t$ (frente a la Normal) para absorber las colas pesadas y los eventos extremos del Brent.
- La estructura logarítmica y la inclusión del término de asimetría permitieron capturar el efecto apalancamiento (las caídas de precio elevaron la volatilidad en mayor medida que las subidas), manteniendo la estacionariedad.
- El modelo EGARCH no logró mantener su capacidad predictiva fuera de la muestra de entrenamiento: la pérdida QLIKE superó el umbral establecido y los residuos conservaron efectos ARCH a corto plazo.
- A pesar de su flexibilidad asimétrica, la forma funcional paramétrica del EGARCH subestimó la magnitud del shock extremo de 2026, evidenciando una rigidez estructural ante cambios abruptos de régimen.
:::

## Introducción

El modelo GARCH de Bollerslev (1986) presenta dos limitaciones estructurales: requiere restricciones de no negatividad sobre sus parámetros para garantizar varianzas positivas y responde de forma simétrica a los choques positivos y negativos de igual magnitud, dado que el signo se pierde al elevar los retornos al cuadrado. 

Para superar ambas limitaciones, Nelson (1991) propone el modelo GARCH Exponencial (EGARCH). Esta especificación modela la dinámica sobre el logaritmo de la varianza condicional, lo que garantiza $\sigma_t^2 > 0$ sin restricciones paramétricas. Además, incorpora un término que depende del signo del choque, permitiendo capturar respuestas asimétricas de la volatilidad.

Esta sección presenta, estima y evalúa el modelo EGARCH bajo el mismo protocolo experimental empleado para el GARCH, con el fin de que sus resultados sean comparables.

### Modelo EGARCH

La varianza condicional se modela según la especificación EGARCH($p, q$):

$$
\ln \sigma_t^2 = \omega + \sum_{i=1}^{p} \left[ \alpha_i \left( |z_{t-i}| - \mathbb{E}|z| \right) + \gamma_i z_{t-i} \right] + \sum_{j=1}^{q} \beta_j \ln \sigma_{t-j}^2
$$

Donde $z_{t-i} = \varepsilon_{t-i} / \sigma_{t-i}$ representa la innovación estandarizada. Cada término cumple una función específica:

- **Efecto magnitud ($\alpha_i$)**: La desviación de $|z_{t-i}|$ respecto a su valor esperado incrementa la log-varianza cuando el choque es mayor de lo habitual.
- **Efecto signo ($\gamma_i$)**: Introduce la asimetría. Un choque negativo ($z < 0$) tiene un impacto $\alpha_i - \gamma_i$ y uno positivo un impacto $\alpha_i + \gamma_i$. Un valor $\gamma_i < 0$ representa el efecto apalancamiento (*leverage effect*), donde las caídas de precio elevan la volatilidad en mayor medida que las subidas de igual magnitud.
- **Persistencia ($\beta_j$)**: Determina la memoria de la log-varianza. 

La condición de estacionariedad para el modelo EGARCH($p, q$) requiere que el valor absoluto de la suma de los coeficientes de persistencia sea menor que uno:

$$
\left| \sum_{j=1}^{q} \beta_j \right| < 1
$$

La implementación se realiza con la librería `arch` de Python en su versión `8.0.0`. Dicha librería utiliza una convención de nomenclatura específica para los argumentos del modelo: `p` para los términos simétricos, `o` para los asimétricos y `q` para los rezagos de la varianza. El *wrapper* `EGARCHWrapper` mantiene esta convención de forma directa ($p=p$, $o=p$, $q=q$).

En esta implementación, el orden del término asimétrico (`o`) se iguala al orden del término de magnitud (`p`). Matemáticamente, esto implica que el efecto apalancamiento ($\gamma_i$) se evalúa sobre la misma ventana temporal de choques recientes que el efecto de magnitud ($\alpha_i$). Esta decisión se fundamenta en la estructura teórica del EGARCH, donde el signo y la magnitud pertenecen a la misma innovación estandarizada $z_t$, y en la práctica econométrica estándar para evitar la sobreparametrización del modelo y facilitar la convergencia numérica del optimizador.

Dado que la ecuación se define sobre el logaritmo de la varianza, los parámetros $\omega$, $\alpha_i$ y $\gamma_i$ no requieren restricciones de signo.

## Entrenamiento y Optimización

Para la selección de hiperparámetros del modelo EGARCH, se aplica el mismo protocolo experimental detallado en la sección del modelo GARCH. Se ejecuta una réplica bajo el esquema Walk-Forward, utilizando optimización bayesiana (Optuna) para explorar los órdenes $p$ y $q$ en el rango $\{1, 2, 3\}$ y las tres distribuciones para las innovaciones (Normal, $t$ de Student y GED).

Se mantienen idénticos los criterios de control computacional (límite de 50 evaluaciones y parada temprana tras 15 intentos), así como el factor de escalado de los retornos (multiplicación por 100) para garantizar la estabilidad numérica del optimizador MLE. A diferencia del modelo GARCH, en el que algunas configuraciones presentaron fallos de convergencia, la estructura logarítmica del EGARCH tiende a una mayor estabilidad numérica, aunque cualquier trial fallido se penaliza con el mismo criterio. Las varianzas pronosticadas se revierten a su escala original (dividiendo por $100^2$) antes de calcular la pérdida QLIKE, permitiendo su comparación directa con los resultados del modelo GARCH.

In [2]:
def egarch_search_space(trial: optuna.Trial) -> dict:
    return {
        "p": trial.suggest_int("p", 1, 3),
        "q": trial.suggest_int("q", 1, 3),
        "dist": trial.suggest_categorical("dist", ["normal", "t", "ged"]),
        "seed": 42
    }

scorer = VolatilityScorer()

exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=EGARCHWrapper,
    scorer=scorer,
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,                          
    suggest_params_fn=egarch_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

Iniciando EGARCH | Ejecución 1/5 | warmup=1210


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando EGARCH | Ejecución 2/5 | warmup=1220


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando EGARCH | Ejecución 3/5 | warmup=1230


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando EGARCH | Ejecución 4/5 | warmup=1240


  0%|          | 0/50 [00:00<?, ?it/s]

Iniciando EGARCH | Ejecución 5/5 | warmup=1250


  0%|          | 0/50 [00:00<?, ?it/s]

El mecanismo de parada temprana (*Early Stopping*) operó sin alcanzar el límite máximo de 50 evaluaciones, deteniéndose en el trial 32. La búsqueda identificó como configuración óptima al EGARCH(1, 3) con distribución $t$ de Student, lo que evidencia una rápida convergencia del espacio de hiperparámetros hacia esta arquitectura. El mecanismo de poda (*Pruning*) no descartó ninguna configuración (0 podados), lo que indica que todos los modelos evaluados lograron la convergencia numérica desde las primeras particiones, confirmando la estabilidad de la estructura logarítmica del EGARCH y evitando descartes por inestabilidad.

El entrenamiento completo bajo el esquema Walk-Forward totalizó 12.54 segundos, con un tiempo medio de 0.139 segundos por partición. Este costo mantiene la tendencia de alta eficiencia computacional de los modelos paramétricos, permitiendo su estimación y actualización diaria en entornos de producción sin requerir hardware especializado.

In [3]:
resumen_data = []

exec_id = 1
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
best_params_str = "-"
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="EGARCHWrapper_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

        p_val = study.best_params.get('p', '?')
        q_val = study.best_params.get('q', '?')
        best_params_str = f"({p_val}, {q_val})"

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor Config (p, q)": best_params_str,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,"Mejor Config (p, q)",Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
32,32,0,"(1, 3)",0.4731,12.5400,0.1390


El ranking de configuraciones muestra que las cinco mejores opciones presentan una pérdida QLIKE en el rango de 0.473 a 0.481. Las diferencias entre estas configuraciones son menores que la variabilidad entre folds ($\sigma \approx 0.10$), por lo que configuraciones con distinto orden de rezagos y distintas distribuciones alcanzan un desempeño similar en QLIKE. No obstante, a diferencia del modelo GARCH, el top-5 está dominado enteramente por distribuciones de colas pesadas ($t$ de Student y GED), lo cual confirma la necesidad de capturar la asimetría y los eventos extremos característicos del Brent mediante esta estructura logarítmica. 

In [4]:
tt.display_top_results(
    RESULTS_PATH,
    top_n=5,
    n_replicas=1,
    column_names=("p", "q", "Distribución", "QLIKE"),
    align_left_first=False,
)

p,q,Distribución,QLIKE
1,3,t,0.473 ± 0.099
1,2,t,0.476 ± 0.100
1,3,ged,0.480 ± 0.105
2,2,t,0.481 ± 0.092
3,2,t,0.481 ± 0.094


Las dos configuraciones con menor pérdida fueron EGARCH(1,3) y EGARCH(1,2) ambos con una distribución $t$. La diferencia entre ambas $\left(\Delta \text{QLIKE} = 3 \times 10^{-3}\right)$ es dos órdenes de magnitud menor que la variabilidad entre folds ($\sigma \approx 0.10$), por lo que son indistinguibles en desempeño predictivo one-step-ahead.

In [26]:
best_model =EGARCHWrapper(p=1, q=3, dist="t")
best_model.fit(y_train, rs_train)

Un aspecto relevante del proceso de selección es que el orden de rezagos elegido por Optuna para el EGARCH $(p=1, q=3)$ coincide con la estructura del modelo GARCH previamente seleccionado. Esta consistencia independiente sugiere que dicha configuración captura de manera robusta la dinámica de dependencia temporal del Brent, independientemente de la forma funcional de la ecuación de varianza (lineal frente a logarítmica). La diferencia radica en que el EGARCH requiere la distribución $t$ de Student para absorber adecuadamente el riesgo de cola y la asimetría, mientras que el GARCH se mantuvo estable con innovaciones gaussianas. Los coeficientes estimados y los intervalos de confianza (obtenidos mediante bootstrap por bloques) para el modelo EGARCH(1,3) seleccionado se presentan a continuación:

In [6]:
df_params_boot = best_model.get_parameter_table(
    B=2000, decimals=4, block_size=22)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior (95%)",
    "CI Upper": "IC Superior (95%)"
})

tt.display_styled_df(
    df=df_params_boot,
    decimals=4,
    p_value_cols=["p-valor"]
)

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior (95%),IC Superior (95%)
omega,0.0352,0.0109,.001,0.0365,0.0949
alpha[1],0.1747,0.0476,< .001,0.1454,0.3205
gamma[1],-0.0695,0.0182,< .001,-0.0198,0.0764
beta[1],0.9405,0.1919,< .001,0.2178,0.9856
beta[2],0.0000,0.4552,1.000,0.0000,0.7374
beta[3],0.0430,0.3161,.892,0.0000,0.5274
nu,5.1729,0.5773,< .001,3.4664,5.0411


El análisis de los intervalos de confianza obtenidos mediante remuestreo por bloques (2000 iteraciones) sobre el reentrenamiento con la muestra completa revela una estructura diferenciada entre los parámetros. El término constante $\omega$, el coeficiente de magnitud $\alpha_1$ y el coeficiente de persistencia $\beta_1$ presentan significancia estadística e intervalos estrictamente positivos. El parámetro de asimetría $\gamma_1$ es negativo y altamente significativo, confirmando la presencia del efecto apalancamiento en la volatilidad del Brent (las caídas de precio elevan la volatilidad en mayor medida que las subidas). Por su parte, el parámetro de grados de libertad $\nu$ es significativo, justificando el uso de la distribución $t$ de Student. El coeficiente $\beta_2$ se reduce a cero, mientras que $\beta_3$ presenta un valor positivo pero no significativo.

Desde la perspectiva clásica, la reducción de $\beta_2$ hacia la frontera inferior y los altos errores estándar de los términos de persistencia de orden superior evidencian una identificación débil bajo el optimizador de Máxima Verosimilitud (MLE). Al igual que en el modelo GARCH, el optimizador desplaza los coeficientes hacia el límite inferior para maximizar la verosimilitud, mientras que $\beta_1$ absorbe la mayor parte de la persistencia. Esta interpretación está acotada al reentrenamiento con la muestra completa de 2171 observaciones. Durante el esquema de ventana expansiva (*Walk-Forward*), el modelo se reestima en cada partición con tamaños crecientes, por lo que $\beta_2$ no está restringido a cero en todos los pasos del pronóstico. La pérdida QLIKE reportada en la clasificación corresponde al desempeño del modelo durante la validación, donde se utiliza la estructura completa.

La condición de estacionariedad para el EGARCH se verifica a partir del valor absoluto de la suma de los coeficientes de persistencia:

$$
\left| \sum_{j=1}^{q} \beta_j \right| = |0.9405 + 0.0000 + 0.0430| = 0.9835 < 1
$$

Dado que el resultado es inferior a 1.0, la log-varianza incondicional es finita y los efectos de los choques se disipan a largo plazo. La proximidad de este valor a la raíz unitaria evidencia una persistencia prolongada en la volatilidad. La vida media de los choques se estima en aproximadamente 42 días hábiles, un periodo más corto que el observado en el GARCH, lo que refleja la mayor velocidad de reacción de la estructura logarítmica ante los cambios de régimen.

Para evaluar la estructura de los residuos dentro de la muestra de entrenamiento, se aplicaron pruebas de diagnóstico. Las pruebas de Ljung-Box (autocorrelación), ARCH-LM (heterocedasticidad) y BDS (dependencia no lineal) no rechazan la hipótesis nula (*p* > .05). Lo anterior indica que el modelo captura la dependencia temporal y los efectos condicionales relevantes.

In [7]:
std_residuals = best_model.model_fit.std_resid

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),7.4431,.683,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),24.5282,.220,.660,Sin Autocorrelación
Ljung-Box (Lag 30),34.2710,.270,.810,Sin Autocorrelación
ARCH-LM (Lag 10),5.7921,.832,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),9.2134,.980,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),19.9643,.917,1.000,Sin Efectos ARCH
Jarque-Bera,1633.6707,< .001,--,Residuos No Normales
Lilliefors,0.0478,< .001,--,Residuos No Normales
BDS (Dim 2),0.4305,.667,--,Sin Estructura No Lineal


Las pruebas de normalidad aplicadas a los residuos (Jarque-Bera y Lilliefors) rechazan la hipótesis nula de Normalidad ($p < .001$). Este resultado es el esperado bajo la especificación seleccionada, dado que el modelo EGARCH asume explícitamente una distribución $t$ de Student para capturar las colas pesadas inherentes a los retornos del Brent. 

Al observar el gráfico Q-Q, se nota un patrón similar al del modelo GARCH: los cuantiles empíricos se alinean de forma precisa en el centro, pero en los extremos (colas) los puntos se desvían de la recta teórica. Esto indica que, aunque la distribución $t$ describe adecuadamente el cuerpo de los residuos, persisten eventos extremos que la distribución paramétrica no logra absorber en su totalidad. A pesar de estas desviaciones visuales, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .6719$; Cramér-von Mises: $p = .3786$) no rechazan la hipótesis de pertenencia a la distribución $t$.

In [11]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

Un aspecto relevante surge al comparar los grados de libertad estimados. El ajuste de la distribución $t$ a los residuos del EGARCH arroja un valor de $\nu \approx 5.07$, el cual es muy cercano al estimado directamente por el modelo ($\nu = 5.17$). Ambos valores son superiores al observado en los residuos en la muestra de entrenamiento del modelo GARCH ($\nu \approx 4.81$). Este incremento en los grados de libertad implica colas más ligeras y una menor concentración de valores extremos. Este comportamiento es consistente con la hipótesis de que la transformación logarítmica contribuye a estabilizar la dinámica de la varianza condicional. Al modelar el logaritmo, es plausible que el impacto de los choques extremos sobre la varianza se atenúe, lo que podría estar traduciéndose en una menor severidad de las colas en los residuos estandarizados y en un acercamiento hacia la Normalidad en comparación con el modelo GARCH lineal.

La prueba de Razón de Verosimilitud (LRT) rechaza la hipótesis nula de igualdad de ajuste ($p < .001$). Asimismo, los criterios de información evidencian una reducción de $\Delta\text{AIC} = 206.85$ y $\Delta\text{BIC} = 201.16$ a favor de la distribución $t$ de Student. Esto apoya la inclusión del parámetro de grados de libertad mejora la descripción de los residuos en la muestra de entrenamiento, justificando su selección sin penalizar en exceso la complejidad del modelo.

In [12]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,089.24","6,182.48","6,193.84",-,-
t-Student,"-2,984.82","5,975.63","5,992.68",208.84,< .001


Al igual que en el modelo GARCH, la volatilidad condicional estimada por el EGARCH sigue la trayectoria general de los clústeres de volatilidad realizados a lo largo del periodo de estudio. El modelo logra capturar la dinámica habitual, reflejando los aumentos y disminuciones graduales en los periodos de relativa calma.

Durante la crisis del COVID-19 se observa un comportamiento idéntico al del modelo GARCH: una subestimación de la magnitud del pico máximo. Mientras que la volatilidad realizada supera el 0.25 durante el colapso de marzo de 2020, la predicción del EGARCH apenas alcanza el 0.10. Este comportamiento sugiere que, a pesar de la flexibilidad adicional que otorgan la transformación logarítmica y el término de asimetría, la estructura paramétrica del EGARCH mantiene una rigidez ante shocks de esta magnitud. La función matemática del modelo suaviza los cambios extremos de régimen, lo que limita su capacidad para replicar los saltos abruptos de la volatilidad empírica.

In [10]:
rs_volatilidad = np.sqrt(rs_train)
pred_volatilidad = best_model.model_fit.conditional_volatility / 100.0
mask_crisis = (df_train.index >= "2020-01-01") & (df_train.index <= "2020-06-30")

ts.plot_volatility_fit(
    dates=df_train.index,
    rs_variance=rs_volatilidad,      
    pred_variance=pred_volatilidad,
    title="Ajuste dentro la Muestra de Entrenamiento (EGARCH 1,3)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad EGARCH $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14)

## Evaluación

La evaluación del EGARCH sigue el mismo protocolo experimental detallado en la sección del modelo GARCH. El rendimiento se contrasta comparando el ajuste sobre los datos observados con la capacidad predictiva sobre datos no vistos. 

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se mantiene el criterio de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Al igual que en el modelo anterior, un valor de $|\Delta_{abs}|$ que supere el umbral de tolerancia de 0.05 evidencia falta de generalización. 

Las predicciones fuera de muestra se generan ejecutando el método `forecast`, el cual aplica un esquema de ventana expansiva (*rolling forward*) para garantizar la ausencia de fuga de información. Las varianzas pronosticadas se devuelven a su escala original para permitir su comparación directa con la volatilidad realizada.

In [27]:
preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test)

df_preds = pd.DataFrame({
    "date": df_test.index, "rs_true": rs_test, "y_pred": preds_test})

output_path = RESULTS_PATH / "best_garch_predictions.csv"
df_preds.to_csv(output_path, index=False)

A partir de las varianzas predichas y los retornos reales del conjunto de prueba, se construyeron los residuos estandarizados fuera de muestra mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

Los resultados del diagnóstico presentaron un contraste parcial con la muestra de entrenamiento. Las pruebas de Ljung-Box (autocorrelación) y BDS (dependencia no lineal) no rechazaron sus hipótesis nulas, lo que indicó que el modelo logró capturar la estructura temporal de largo plazo. Sin embargo, la prueba ARCH-LM detectó efectos ARCH significativos en el rezago 10 ($p\text{-adj} = .005$), aunque no en los rezagos 20 y 30. 

In [28]:
rs_test_vol = df_preds["rs_true"]
pred_test_vol = df_preds["y_pred"]

std_residuals_oos = y_test / np.sqrt(df_preds["y_pred"])
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)
df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),13.7573,.184,.553,Sin Autocorrelación
Ljung-Box (Lag 20),28.7691,.092,.277,Sin Autocorrelación
Ljung-Box (Lag 30),32.3650,.351,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),28.2295,.002,.005,Efectos ARCH Significativos
ARCH-LM (Lag 20),32.7559,.036,.108,Sin Efectos ARCH
ARCH-LM (Lag 30),40.0886,.103,.309,Sin Efectos ARCH
Jarque-Bera,273.7582,< .001,--,Residuos No Normales
Lilliefors,0.0427,< .001,--,Residuos No Normales
BDS (Dim 2),0.2099,.834,--,Sin Estructura No Lineal


Este comportamiento contrasta con el del modelo GARCH lineal, el cual no presentó efectos ARCH remanentes en el conjunto de prueba. Una hipótesis plausible para explicar esta diferencia radica en la respuesta de cada arquitectura ante los shocks de corto plazo. Es posible que el GARCH lineal, al reaccionar de forma directa a las innovaciones al cuadrado recientes (términos $\alpha$), haya logrado absorber y descontar con mayor rapidez los clústeres de volatilidad repentinos. En contraste, la transformación logarítmica del EGARCH comprimió la magnitud de los choques. Aunque esto le permitió capturar el efecto apalancamiento y mantener una alta persistencia (con un $\beta_1 = 0.9405$), también pudo haber provocado que el modelo reaccionara de manera más lenta o "suave" ante picos abruptos.

Dado que el EGARCH subestimó la magnitud del shock extremo de 2026, es probable que los errores de predicción en ese periodo conservaran una estructura de dependencia condicional de corto plazo (efectos ARCH) que el modelo no logró filtrar. Esta incapacidad para adaptarse a la dinámica de shocks rápidos evidenció que, a pesar de su flexibilidad asimétrica, la estructura paramétrica del EGARCH seguía siendo rígida para ciertos regímenes. Este hallazgo justificó empíricamente la exploración de arquitecturas no paramétricas con mayor flexibilidad, capaces de aprender patrones complejos sin asumir una forma funcional predefinida.

Los residuos fuera de muestra mantuvieron un ajuste adecuado a la distribución $t$ de Student (Kolmogorov-Smirnov: $p = .5813$; Cramér-von Mises: $p = .3452$). El parámetro de grados de libertad estimado fue $\nu \approx 5.35$, superior al observado en el entrenamiento ($\nu \approx 5.07$). Este incremento indicó colas más ligeras y un mayor acercamiento a la distribución gaussina, consistente con la menor severidad de los shocks en el conjunto de prueba. La prueba de Razón de Verosimilitud $\left(\text{LRT} = 73.47, p < .001\right)$ y los criterios de información ($\Delta\text{AIC} = 71.47$ y $\Delta\text{BIC} = 66.61$) confirmaron la superioridad estadística de la distribución $t$ frente a la Normal.

In [17]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

El modelo EGARCH(1,3) no cumplió con los criterios de adecuación pre-fijados para las métricas principales. La pérdida QLIKE superó ampliamente el umbral de tolerancia absoluto establecido $\left(\Delta = +0.1252 > 0.05\right)$, lo que evidenció una falta de generalización fuera de la muestra de entrenamiento. No obstante, las métricas de error puntual (MSE, RMSE, MAE) y probabilísticas (CRPS y Pinball) presentaron variaciones marginales. 

In [30]:
preds_train_var = (best_model.model_fit.conditional_volatility ** 2) / 10000.0
preds_test_var = df_preds["y_pred"].values

ts.display_train_test_comparison(
    rs_train=rs_train,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.3816,0.5068,+0.1252
CRPS,0.0126,0.0123,-0.0002
Pinball (τ=0.05),0.0029,0.0028,-0.0001
Pinball (τ=0.95),0.0014,0.0017,+0.0003
MSE,0.0000,0.0000,+0.0000
RMSE,0.0018,0.0022,+0.0004
MAE,0.0004,0.0005,+0.0000
HMSE,1.3491,10.1008,+8.7517
HMAE,0.6329,0.7719,+0.1390


Las métricas ajustadas por heterocedasticidad superaron el umbral establecido, con incrementos de $+8.7517$ en el HMSE y $+0.1390$ en el HMAE. Este resultado es consistente con la definición del HMSE, el cual pondera los errores por el inverso de la volatilidad. Esto magnificó el error de predicción durante los días de mayor turbulencia en el conjunto de prueba.

Este incremento reflejó una limitación específica ante el régimen de 2026. Al no capturar la magnitud del pico extremo, el modelo generó errores elevados en esos días. Este deterioro no indicó sobreajuste, dado que el modelo es parsimonioso. En cambio, apuntó hacia una rigidez estructural: la forma funcional paramétrica del EGARCH no se adaptó a los cambios abruptos de régimen.

In [21]:
rs_test_vol = np.sqrt(df_preds["rs_true"])
pred_test_vol = np.sqrt(df_preds["y_pred"])
mask_crisis = (df_preds["date"] >= "2026-02-01") & (df_preds["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste fuera la Muestra de Entrenamiento (GARCH 1,3)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad GARCH $\left(\hat{\sigma}_t\right)$",
    
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Conclusiones

La implementación y evaluación del modelo EGARCH(1,3) permitió contrastar la capacidad de las especificaciones paramétricas asimétricas para el pronóstico de la volatilidad del Brent. A diferencia del modelo GARCH, los diagnósticos sobre los residuos fuera de muestra evidenciaron efectos ARCH remanentes a corto plazo, y los criterios de adecuación no se cumplieron (el QLIKE superó el umbral pre-fijado). Esto indicó un fallo de generalización de la estructura logarítmica ante el régimen de mercado del conjunto de prueba.

La estructura EGARCH(1,3) modela la dinámica sobre el logaritmo de la varianza condicional e incorpora un término de asimetría. Esta configuración permitió capturar el efecto apalancamiento esperado en los retornos del petróleo, donde las caídas de precio tienden a elevar la volatilidad en mayor medida que las subidas de igual magnitud. 

Los análisis revelaron limitaciones estructurales de esta especificación. A pesar de que el modelo incorporó una distribución $t$ de Student y un término de asimetría para absorber las colas pesadas y el efecto apalancamiento, el incremento severo en las métricas ajustadas por heterocedasticidad (HMSE y HMAE) reflejó dificultades para adaptarse a la magnitud de los shocks extremos. Esto indica que la transformación logarítmica, si bien otorga flexibilidad teórica, suavizó en exceso la reacción del modelo frente a los cambios abruptos de régimen, como el pico de 2026.

Estos hallazgos motivan la transición hacia arquitecturas con mayor flexibilidad. La evidencia empírica demostró que, a pesar de la incorporación de mecanismos de asimetría, las formas funcionales paramétricas mantienen un techo estructural para capturar la dinámica de shocks abruptos. Esto sustenta la necesidad de evaluar arquitecturas no paramétricas capaces de aprender relaciones no lineales complejas (como las redes neuronales LSTM y GRU) sin asumir una forma matemática predefinida.